In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


# =========================================================
# 1. TOOL
# =========================================================

@tool
def delete_report(report_id: str) -> str:
    """
    Delete a report.
    This is a simulated operation for learning.
    """

    print()
    print(">>> DELETE REPORT TOOL EXECUTED")
    print("Report:", report_id)

    return f"Report {report_id} deleted successfully."


# =========================================================
# 2. MODEL
# =========================================================

model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)


# =========================================================
# 3. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()


# =========================================================
# 4. HUMAN-IN-THE-LOOP
# =========================================================

hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "delete_report": {
            "allowed_decisions": [
                "approve",
                "reject",
            ],
            "description": (
                "Deleting a report requires human approval."
            ),
        }
    }
)


# =========================================================
# 5. AGENT
# =========================================================

agent = create_agent(
    model=model,

    tools=[
        delete_report,
    ],

    middleware=[
        hitl,
    ],

    checkpointer=checkpointer,
)


# =========================================================
# 6. THREAD
# =========================================================

config = {
    "configurable": {
        "thread_id": "conversation-001",
    }
}


# =========================================================
# 7. USER REQUEST
# =========================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "گزارش 123 را حذف کن.",
            }
        ]
    },
    config,
)


# =========================================================
# 8. CHECK INTERRUPT
# =========================================================

print()
print("========================================")
print("AGENT RESULT")
print("========================================")

print(result)


# =========================================================
# 9. HUMAN DECISION
# =========================================================

if "__interrupt__" in result:

    print()
    print("========================================")
    print("HUMAN APPROVAL REQUIRED")
    print("========================================")

    decision = input(
        "Approve deletion? (yes/no): "
    )


    # =====================================================
    # APPROVE
    # =====================================================

    if decision.lower() == "yes":

        result = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "approve"
                        }
                    ]
                }
            ),
            config,
        )


    # =====================================================
    # REJECT
    # =====================================================

    else:

        result = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "reject",
                            "message": (
                                "Human rejected the deletion."
                            ),
                        }
                    ]
                }
            ),
            config,
        )


# =========================================================
# 10. FINAL RESULT
# =========================================================

print()
print("========================================")
print("FINAL RESULT")
print("========================================")

print(
    result["messages"][-1].content
)